In [52]:
# data load
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loading import load_raw_data

df = load_raw_data("../data/raw/raw_credit_applications.json")
df.shape

(502, 34)

### 1. Completeness
In this section, the completeness of the dataset is analyzed by assessing the presence of missing values across all relevant fields. The objective is to identify incomplete records that may affect downstream analyses or decision-making.

In [53]:
# count missing values
# exclude spending columns
cols_no_spending = [c for c in df.columns if not c.startswith("spending_")]

missing = df[cols_no_spending].isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing

notes                   500
loan_purpose            452
processing_timestamp    440
rejection_reason        292
approved_amount         210
interest_rate           210
ssn                       5
ip_address                5
annual_income             5
zip_code                  1
gender                    1
date_of_birth             1
dtype: int64

In [54]:
# percentage
(missing / len(df) * 100).round(2)

notes                   99.60
loan_purpose            90.04
processing_timestamp    87.65
rejection_reason        58.17
approved_amount         41.83
interest_rate           41.83
ssn                      1.00
ip_address               1.00
annual_income            1.00
zip_code                 0.20
gender                   0.20
date_of_birth            0.20
dtype: float64

In [55]:
# approved but missing required decision fields
bad_approved = df.query("loan_approved == True and (approved_amount.isna() or interest_rate.isna())")
len(bad_approved)

0

In [56]:
# rejected but has approval fields filled (should be empty)
bad_rejected = df.query("loan_approved == False and (approved_amount.notna() or interest_rate.notna())")
len(bad_rejected)

0

In [57]:
# check wheter spending information is present at the record level
spend_cols = [c for c in df.columns if c.startswith("spending_")]

no_spending = df[spend_cols].isna().all(axis=1).sum()
has_spending = len(df) - no_spending

no_spending, has_spending, no_spending / len(df)

(np.int64(0), np.int64(502), np.float64(0.0))

In [58]:
'''
he record-level analysis shows that all 502 applications contain at least some spending information, indicating no completeness issues for the pivoted spending fields. However, high missingness in loan_purpose and processing_timestamp suggests structural data collection gaps, while notes appears optional. Decision-related fields display conditional missingness aligned with approval status, but inconsistencies should be flagged. Critical financial variables such as annual_income require explicit flagging and potential exclusion from modeling rather than blind imputation.

Option A - Flag only:
df["flag_missing_income"] = df["annual_income"].isna()

Option B - Exclude from modeling:
df_model = df[df["annual_income"].notna()]

'''

'\nhe record-level analysis shows that all 502 applications contain at least some spending information, indicating no completeness issues for the pivoted spending fields. However, high missingness in loan_purpose and processing_timestamp suggests structural data collection gaps, while notes appears optional. Decision-related fields display conditional missingness aligned with approval status, but inconsistencies should be flagged. Critical financial variables such as annual_income require explicit flagging and potential exclusion from modeling rather than blind imputation.\n\nOption A - Flag only:\ndf["flag_missing_income"] = df["annual_income"].isna()\n\nOption B - Exclude from modeling:\ndf_model = df[df["annual_income"].notna()]\n\n'

### 2. Consistency
In this section, internal consistency is evaluated by examining whether records are logically coherent and structurally unique. This includes identifying duplicate application IDs and detecting contradictions between related fields that may compromise data integrity.

In [59]:
# duplicates
df.duplicated().sum()


np.int64(0)

In [60]:
# id based duplicates
df["id"].duplicated().sum()

np.int64(2)

In [61]:
dup_ids = df.loc[df["id"].duplicated(keep=False), ["id", "processing_timestamp", "email", "ssn", "loan_approved", "approved_amount", "interest_rate"]]
dup_ids.sort_values(["id", "processing_timestamp"])

,id,processing_timestamp,email,ssn,loan_approved,approved_amount,interest_rate
383,app_001,None,stephanie.nguyen47@mail.com,427-90-1892,False,NaN,NaN
455,app_001,None,stephanie.nguyen47@mail.com,None,False,NaN,NaN
8,app_042,None,joseph.lopez1@gmail.com,652-70-5530,False,NaN,NaN
354,app_042,None,joseph.lopez1@gmail.com,652-70-5530,False,NaN,NaN


In [62]:
if "gender" in df.columns:
    print("gender")
    print(df["gender"].value_counts(dropna=False))

gender
gender
Male      195
Female    193
F          58
M          53
            2
None        1
Name: count, dtype: int64


In [63]:
'''
The consistency analysis identified 2 duplicate application IDs, while no full row-level duplicates were detected. Although the duplicated records appear largely similar, repeated IDs violate the uniqueness requirement and may compromise data integrity and traceability.
In addition, inconsistent categorical encoding was detected in the gender field. The same logical categories are represented using multiple labels (e.g., “Male” and “M”, “Female” and “F”), indicating a lack of standardized data entry. Such inconsistencies may distort aggregation results and bias downstream analysis.
Remediation should either flag duplicate IDs for review or deduplicate based on a defined business rule (e.g., keeping the first or most recent record). Furthermore, categorical values should be standardized to ensure consistent encoding.

Option A - Flag duplicates:
df["flag_duplicate_id"] = df["id"].duplicated(keep=False)

Option B - Remove duplicates:
df_deduplicated = df.drop_duplicates(subset="id", keep="first")

Standardize gender encoding:
df["gender"] = df["gender"].replace({
    "M": "Male",
    "F": "Female",
    "": pd.NA
})
'''

'\nThe consistency analysis identified 2 duplicate application IDs, while no full row-level duplicates were detected. Although the duplicated records appear largely similar, repeated IDs violate the uniqueness requirement and may compromise data integrity and traceability.\nIn addition, inconsistent categorical encoding was detected in the gender field. The same logical categories are represented using multiple labels (e.g., “Male” and “M”, “Female” and “F”), indicating a lack of standardized data entry. Such inconsistencies may distort aggregation results and bias downstream analysis.\nRemediation should either flag duplicate IDs for review or deduplicate based on a defined business rule (e.g., keeping the first or most recent record). Furthermore, categorical values should be standardized to ensure consistent encoding.\n\nOption A - Flag duplicates:\ndf["flag_duplicate_id"] = df["id"].duplicated(keep=False)\n\nOption B - Remove duplicates:\ndf_deduplicated = df.drop_duplicates(subset

### 3. Validity
In this section, data validity is assessed by verifying whether values comply with predefined formats, ranges, and business rules. This includes checking for invalid formats (e.g., email, ZIP, SSN) and out-of-bound numerical values such as negative income or implausible financial ratios.

In [64]:
import pandas as pd

income = pd.to_numeric(df["annual_income"], errors="coerce")

# negative income
neg_income = (income < 0).sum()

# debt_to_income outside [0,1]
bad_dti = ((df["debt_to_income"] < 0) | (df["debt_to_income"] > 1)).sum()

# negative interest rate
neg_ir = (df["interest_rate"] < 0).sum()

neg_income, bad_dti, neg_ir

(np.int64(0), np.int64(1), np.int64(0))

In [65]:
#check format related validity issues
invalid_income_format = df["annual_income"].notna().sum() - income.notna().sum()
invalid_income_format

np.int64(0)

In [66]:
# negative spending
neg_spending = (df[spend_cols] < 0).sum().sum()

# extreme values (z.B. > 1 Mio)
extreme_spending = (df[spend_cols] > 1_000_000).sum().sum()

neg_spending, extreme_spending

(np.int64(0), np.int64(0))

In [67]:
# Validate that SSN values follow the required XXX-XX-XXXX format
import re

invalid_ssn = df["ssn"].dropna().apply(
    lambda x: not bool(re.fullmatch(r"\d{3}-\d{2}-\d{4}", str(x)))
).sum()

invalid_ssn

np.int64(0)

In [68]:
# Validate that ZIP codes contain only numeric characters
invalid_zip = df["zip_code"].dropna().apply(
    lambda x: not str(x).isdigit()
).sum()

invalid_zip


np.int64(1)

In [69]:
# Validate that email values contain the required "@" symbol 
invalid_email = df["email"].dropna().apply(
    lambda x: "@" not in str(x)
).sum()

invalid_email

np.int64(8)

In [70]:
'''
The validity analysis identified 1 record with a debt-to-income ratio outside the valid range [0,1], 1 ZIP code containing non-numeric characters, and 8 email addresses missing the required "@" symbol. No negative income values (0), negative interest rates (0), or SSN format violations (0) were detected. 

In total, 10 rule-based validity violations were observed. Although the number is low relative to the dataset size, these records should be explicitly flagged or removed to ensure compliance with defined business constraints.

Option A - Flag invalid values:
df["flag_invalid_dti"] = (df["debt_to_income"] < 0) | (df["debt_to_income"] > 1)
df["flag_invalid_zip"] = ~df["zip_code"].astype(str).str.isdigit()
df["flag_invalid_email"] = ~df["email"].astype(str).str.contains("@", na=False)

Option B - Filter to valid values only:
df_valid = df[
    (df["debt_to_income"] >= 0) & (df["debt_to_income"] <= 1) &
    (df["zip_code"].astype(str).str.isdigit()) &
    (df["email"].astype(str).str.contains("@", na=False))
]
'''

'\nThe validity analysis identified 1 record with a debt-to-income ratio outside the valid range [0,1], 1 ZIP code containing non-numeric characters, and 8 email addresses missing the required "@" symbol. No negative income values (0), negative interest rates (0), or SSN format violations (0) were detected. \n\nIn total, 10 rule-based validity violations were observed. Although the number is low relative to the dataset size, these records should be explicitly flagged or removed to ensure compliance with defined business constraints.\n\nOption A - Flag invalid values:\ndf["flag_invalid_dti"] = (df["debt_to_income"] < 0) | (df["debt_to_income"] > 1)\ndf["flag_invalid_zip"] = ~df["zip_code"].astype(str).str.isdigit()\ndf["flag_invalid_email"] = ~df["email"].astype(str).str.contains("@", na=False)\n\nOption B - Filter to valid values only:\ndf_valid = df[\n    (df["debt_to_income"] >= 0) & (df["debt_to_income"] <= 1) &\n    (df["zip_code"].astype(str).str.isdigit()) &\n    (df["email"].ast

### 4. Accuracy
In this section, accuracy is assessed by evaluating whether recorded values plausibly reflect real-world conditions and logical business relationships. This includes detecting implausible age values, unrealistic spending relative to income, and approvals granted despite zero or missing income, in order to identify records that may be factually incorrect even if they pass basic format and validity checks.

In [71]:
#Accuracy Age
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], errors="coerce")
df["age"] = (pd.Timestamp("today") - df["date_of_birth"]).dt.days // 365

unrealistic_age = ((df["age"] < 18) | (df["age"] > 100)).sum()
unrealistic_age

np.int64(0)

In [72]:
#Accuracy Spending > Income?
spend_cols = [c for c in df.columns if c.startswith("spending_")]
df["total_spending"] = df[spend_cols].sum(axis=1)

income = pd.to_numeric(df["annual_income"], errors="coerce")

implausible_spending = (df["total_spending"] > income).sum()
implausible_spending

np.int64(1)

In [73]:
#Accuracy Approved loans with Zero or missing income

approved_zero_income = (
    (df["loan_approved"] == True) &
    ((income <= 0) | income.isna())
).sum()

approved_zero_income

np.int64(3)

In [74]:
"""
The accuracy analysis identified 0 unrealistic age values (<18 or >100), 
1 case where total spending exceeds reported annual income, and 
3 approved loans with zero or missing income.

In total, 4 records show plausibility-related accuracy concerns that may reflect 
reporting errors or decision inconsistencies and should be reviewed.

Option A - Flag implausible cases:
df["flag_implausible_spending"] = df["total_spending"] > income
df["flag_approved_zero_income"] = (df["loan_approved"] == True) & ((income <= 0) | income.isna())

Option B - Exclude from modeling:
df_model = df[
    ~(df["total_spending"] > income) &
    ~((df["loan_approved"] == True) & ((income <= 0) | income.isna()))
]
"""

'\nThe accuracy analysis identified 0 unrealistic age values (<18 or >100), \n1 case where total spending exceeds reported annual income, and \n3 approved loans with zero or missing income.\n\nIn total, 4 records show plausibility-related accuracy concerns that may reflect \nreporting errors or decision inconsistencies and should be reviewed.\n\nOption A - Flag implausible cases:\ndf["flag_implausible_spending"] = df["total_spending"] > income\ndf["flag_approved_zero_income"] = (df["loan_approved"] == True) & ((income <= 0) | income.isna())\n\nOption B - Exclude from modeling:\ndf_model = df[\n    ~(df["total_spending"] > income) &\n    ~((df["loan_approved"] == True) & ((income <= 0) | income.isna()))\n]\n'